In [1]:
# Databricks notebook source
# 60 TPD melter - SFC recommender: INFERENCE + RECOMMENDATION E-MAIL JOB (2 of 2)
#
# 1. loads the model released by 01_training_sfc_model (OUTPUT folder, sfc_recommender_latest.json)
# 2. reads the latest plant values from the INPUT folder (or takes them from the MANUAL cell)
# 3. computes the setpoints the operators should set: NG flow, secondary air, air-fuel ratio
#    (next reversal cycle) and barrier boost (next hour)
# 4. checks every setpoint and input against the furnace's historical limits (P1-P99)
# 5. builds the recommendation e-mail, previews it, and posts it to the plant's Logic App
#
# SEND_MAIL is False by default: the message is rendered in the notebook so it can be checked
# before anything leaves the workspace. Schedule hourly (boost) or every reversal cycle (gas).

# SFC Recommender — Inference, Recommendation and E-mail
**60 TPD flint melter · what the operators should set now**

| | |
|---|---|
| **Reads** | `output/sfc_recommender_latest.json` (model from the training job) and `input/analytical_record_15min.csv` (latest plant values) |
| **Shows** | recommended NG flow, secondary air, air-fuel ratio and barrier boost, each checked against the historical range |
| **Sends** | the recommendation e-mail through the plant's Logic App (last cell, only when `SEND_MAIL = True`) |
| **Writes** | `output/recommendation_log.csv` (one row per issued recommendation, for the model-vs-actual review) |

In [2]:
import json
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import requests

try:                                   # Databricks
    displayHTML                        # noqa: B018
except NameError:                      # local Jupyter
    from IPython.display import HTML, display
    def displayHTML(html):             # noqa: N802
        display(HTML(html))

## 1. Settings
Everything the job needs is in this cell. Choose where the plant values come from with `INPUT_MODE`:
- `"latest_data"`: the most recent valid values in the input file / table
- `"manual"`: the values typed in `MANUAL_INPUTS`

In [3]:
# dbutils.widgets.text("input_dir",  "/Volumes/prod_pgp_mfg/cantier/60_tpd_sfc/input")
# dbutils.widgets.text("output_dir", "/Volumes/prod_pgp_mfg/cantier/60_tpd_sfc/output")
# dbutils.widgets.dropdown("send_mail", "false", ["false", "true"])

# ---- folders -------------------------------------------------------------------------------
INPUT_DIR = Path("input")                        # e.g. Path("/Volumes/prod_pgp_mfg/cantier/60_tpd_sfc/input")
OUTPUT_DIR = Path("output")                      # e.g. Path("/Volumes/prod_pgp_mfg/cantier/60_tpd_sfc/output")
INPUT_FILE = "analytical_record_15min.csv"       # .csv or .parquet (same layout as the training input)
MODEL_FILE = "sfc_recommender_latest.json"

# ---- where the plant values come from ------------------------------------------------------
INPUT_MODE = "latest_data"                       # "latest_data" | "manual"
MANUAL_INPUTS = dict(
    timestamp="2026-10-06 10:00",                # time the values were read
    ncv=9300,                                    # latest valid NCV, kcal/SCM
    draw_t=58.0,                                 # expected draw today, t/day
    cullet_pct=18,                               # cullet % today
    optical_prev24h=1574.5,                      # optical crown temperature, average of the last 24 h, degC
    bb_kwh_last_hour=350,                        # barrier boost in the last hour, kWh
    mb_kwh_last_hour=300,                        # melter boost in the last hour, kWh
    mb3_now=1321.5,                              # MB3 bottom temperature now, degC
    ng_now=None, air_now=None,                   # optional: what is set now (shown as "current")
)
MAX_AGE_MINUTES = {"ncv": 30, "boost": 120, "mb3": 60, "optical": 26 * 60}   # older inputs are flagged

# ---- mail ----------------------------------------------------------------------------------
SEND_MAIL = False          # False = build and preview the message only, nothing is sent
# The plant's Logic App. Move it to dbutils.secrets.get(scope=..., key=...) before this becomes a job.
MAIL_URL = (
    "https://prod2-08.centralindia.logic.azure.com:443/workflows/"
    "6bbc5327ebdf4290b4d40c4eefa5ea8c/triggers/manual/paths/invoke?api-version=2016-10-01"
    "&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=6yF-HEi9KLqLjUVLPOxcDjlTmjLTTIqXsTG-ocF58WQ"
)
MAIL_TIMEOUT = 30
MAIL_SUBJECT_PREFIX = "SFC Recommendation"
PLANT_CAPACITY = "60 TPD"
RECIPIENT_GROUPS = {
    "production": {"to": ["balaji.kesavan@cantier.com"], "cc": ["rudhuvarshan.r@cantier.com"]},
    "quality":    {"to": ["quality.head@example.com"],    "cc": ["process.engineer@example.com"]},
}
MAIL_GROUP = "production"
TO_EMAILS, CC_EMAILS = ["balaji.kesavan@cantier.com"], []
if MAIL_GROUP:
    TO_EMAILS, CC_EMAILS = RECIPIENT_GROUPS[MAIL_GROUP]["to"], RECIPIENT_GROUPS[MAIL_GROUP]["cc"]
print("plant values:", INPUT_MODE, "| mail:", ("WILL BE SENT to " + ", ".join(TO_EMAILS)) if SEND_MAIL else "preview only (SEND_MAIL is False)")

plant values: latest_data | mail: preview only (SEND_MAIL is False)


## 2. Load the model

In [4]:
model = json.loads((OUTPUT_DIR / MODEL_FILE).read_text())
LIM = {k: tuple(v) for k, v in model["limits"].items()}
GAS, AIR, BOOST = model["gas"], model["air"], model["boost"]
MODEL_VERSION = model.get("model_version", model["trained_until"])

pd.DataFrame({"value": {
    "model version": MODEL_VERSION, "trained until": model["trained_until"], "training days": model["n_training_days"],
    "gas: draw (Gcal/day per t)": round(GAS["coefficients"]["draw_t"], 4),
    "gas: barrier boost (Gcal per Gcal)": round(GAS["coefficients"]["bb_g"], 4),
    "gas: furnace age (Gcal/day per month)": round(GAS["coefficients"]["age_m"], 4),
    "gas: efficient-quarter shift (Gcal/day)": round(GAS["conformal_shift"], 3),
    "air per Mcal target (SCM/Mcal)": round(AIR["air_per_mcal"], 4),
    "MB3 target (degC)": round(BOOST["mb3_target"], 2),
    "MB3 gain (degC per +100 kWh/h)": round(BOOST["mb3_gain_degC_per_kwh_h"] * 100, 2)}})

,value
model version,2026-09-29_202610060711
trained until,2026-09-29
training days,343
gas: draw (Gcal/day per t),0.4953
gas: barrier boost (Gcal per Gcal),-0.7715
gas: furnace age (Gcal/day per month),0.3498
gas: efficient-quarter shift (Gcal/day),-0.412
air per Mcal target (SCM/Mcal),1.2676
MB3 target (degC),1320.25
MB3 gain (degC per +100 kWh/h),1.94


## 3. Current plant values
In `latest_data` mode the input is cleaned with the same rules as training, and the most recent **valid** value of each signal is taken:
- **NCV:** the latest reading that passes the range and spike filter
- **Boosts:** the last complete hour (4 × 15-min readings)
- **MB3:** the latest reading
- **Optical:** the average of the last 24 h
- **Draw and cullet:** the latest day's values

Inputs older than `MAX_AGE_MINUTES` are flagged.

In [5]:
COLUMNS = {"Timestamps": "ts", "BARRIER BOOSTER-52": "bb_kwh", "MELTER BOOSTER-51": "mb_kwh", "NCV Meter": "ncv",
           "Cullet %": "cullet_pct", "Melter Optical Temperature": "opt_temp",
           "dcs_60_SecondaryAirFlowvalueFTSAFPVDB205DD92": "sec_air", "dcs_60_MelterGasflowvalueFTMGFPVDB202DD92": "ng_scm",
           "dcs_60_ThermocoupleMelterBottom3TCMB3PVDB249DD572": "mb3_temp", "Daily Draw": "draw_kg", "MB51_Quantity_KG (Draw)": "draw_kg"}
NUMERIC = ["bb_kwh", "mb_kwh", "ncv", "cullet_pct", "opt_temp", "sec_air", "ng_scm", "mb3_temp", "draw_kg"]
VALID_RANGE = {"ncv": (8500, 10500), "opt_temp": (1550, 1600), "mb_kwh": (1, 200), "bb_kwh": (1, 250)}
NCV_SPIKE = 400


def read_table(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path) if str(path).lower().endswith(".parquet") else pd.read_csv(path)


def clean_recent(raw: pd.DataFrame, days: int = 3) -> pd.DataFrame:
    raw = raw.rename(columns=COLUMNS)
    raw["ts"] = pd.to_datetime(raw["ts"])
    raw = raw[raw["ts"] >= raw["ts"].max() - pd.Timedelta(days=days)]
    for c in NUMERIC:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
    q = raw.groupby("ts")[NUMERIC].mean().sort_index()
    for col, (lo, hi) in VALID_RANGE.items():
        q.loc[q[col].notna() & ~q[col].between(lo, hi), col] = np.nan
    med = q["ncv"].rolling(9, center=True, min_periods=3).median()
    q.loc[(q["ncv"] - med).abs() > NCV_SPIKE, "ncv"] = np.nan
    return q


def latest(series: pd.Series):
    s = series.dropna()
    return (float(s.iloc[-1]), s.index[-1]) if len(s) else (np.nan, pd.NaT)


if INPUT_MODE == "latest_data":
    q = clean_recent(read_table(INPUT_DIR / INPUT_FILE))
    ncv, t_ncv = latest(q["ncv"])
    asof = t_ncv
    hb = q[["bb_kwh", "mb_kwh"]].resample("h").sum(min_count=4)
    bb, t_bb = latest(hb["bb_kwh"]); mb, t_mb = latest(hb["mb_kwh"])
    mb3, t_mb3 = latest(q["mb3_temp"])
    opt_win = q.loc[asof - pd.Timedelta(hours=24):asof, "opt_temp"].dropna()
    if opt_win.empty:                                  # no optical in the last 24 h -> last available 24 h
        opt_last = q["opt_temp"].dropna()
        opt_win = opt_last[opt_last.index >= opt_last.index[-1] - pd.Timedelta(hours=24)] if len(opt_last) else opt_last
    optical, t_opt = (float(opt_win.mean()), opt_win.index[-1]) if len(opt_win) else (np.nan, pd.NaT)
    draw_t = latest(q["draw_kg"])[0] / 1000
    cullet = latest(q["cullet_pct"])[0]
    ng_now, _ = latest(q["ng_scm"]); air_now, _ = latest(q["sec_air"])
    inputs = dict(date=asof.normalize(), ncv=ncv, draw_t=draw_t, cullet_pct=cullet, optical_prev24h=optical,
                  bb_kwh_last_hour=bb, mb_kwh_last_hour=mb, mb3_now=mb3)
    stamps = {"ncv": t_ncv, "boost": t_bb + pd.Timedelta(hours=1) if pd.notna(t_bb) else pd.NaT, "mb3": t_mb3, "optical": t_opt}
else:
    m = MANUAL_INPUTS
    asof = pd.Timestamp(m["timestamp"])
    inputs = {k: m[k] for k in ["ncv", "draw_t", "cullet_pct", "optical_prev24h", "bb_kwh_last_hour", "mb_kwh_last_hour", "mb3_now"]}
    inputs["date"] = asof.normalize()
    ng_now, air_now = (m.get("ng_now") or np.nan), (m.get("air_now") or np.nan)
    stamps = {k: asof for k in MAX_AGE_MINUTES}

data_flags = []
for key, t in stamps.items():
    age = (asof - t).total_seconds() / 60 if pd.notna(t) else np.inf
    if age > MAX_AGE_MINUTES[key]:
        what = "last complete hour ended" if key == "boost" else "last valid value at"
        data_flags.append(f"{key} data is old: {what} {t:%d %b %H:%M}" if pd.notna(t) else f"{key}: no valid data")

pd.DataFrame({"value": {
    "recommendation time (latest valid NCV)": f"{asof:%d %b %Y %H:%M}",
    "NCV (kcal/SCM)": round(inputs["ncv"], 0), "draw (t/day)": round(inputs["draw_t"], 2), "cullet (%)": inputs["cullet_pct"],
    "barrier boost, last hour (kWh)": round(inputs["bb_kwh_last_hour"], 1), "melter boost, last hour (kWh)": round(inputs["mb_kwh_last_hour"], 1),
    "MB3 now (degC)": round(inputs["mb3_now"], 2), "optical crown, 24 h average (degC)": round(inputs["optical_prev24h"], 1),
    "data checks": "; ".join(data_flags) or "all inputs fresh"}})

,value
recommendation time (latest valid NCV),30 Sep 2026 23:45
NCV (kcal/SCM),8621.0
draw (t/day),61.1
cullet (%),18.0
"barrier boost, last hour (kWh)",402.2
"melter boost, last hour (kWh)",306.7
MB3 now (degC),1320.88
"optical crown, 24 h average (degC)",1570.1
data checks,boost data is old: last complete hour ended 30...


## 4. Recommendation
The same calculation as the model training (and the backend API):
1. **Daily gas-heat target** = regression (draw, barrier boost, melter boost, furnace age) + efficient-quarter shift
2. **NG** = target ÷ 96 ÷ latest NCV: steady heat, exact NCV compensation
3. **Secondary air** = air-per-Mcal target × gas heat; **AFR** = air ÷ NG
4. **Barrier boost** = last hour + 0.1 × (MB3 target − MB3 now) ÷ gain; it is never reduced while NG is at its upper limit
5. Every setpoint is held inside its historical P1–P99 range

In [6]:
def recommend(inp: dict) -> tuple[pd.DataFrame, list]:
    c = GAS["coefficients"]
    age_m = (pd.Timestamp(inp["date"]) - pd.Timestamp(GAS["age_origin"])).days / GAS["days_per_month"]
    bb_g = inp["bb_kwh_last_hour"] * 24 * GAS["kcal_per_kwh"] / 1e6
    mb_g = inp["mb_kwh_last_hour"] * 24 * GAS["kcal_per_kwh"] / 1e6
    target = (c["Intercept"] + c["draw_t"] * inp["draw_t"] + c["bb_g"] * bb_g + c["mb_g"] * mb_g
              + c["age_m"] * age_m + GAS["conformal_shift"])
    heat = target * 1e6 / GAS["intervals_per_day"]
    ng_raw = heat / inp["ncv"]
    ng = float(np.clip(ng_raw, *LIM["ng_scm"]))
    air_raw = AIR["air_per_mcal"] * ng * inp["ncv"] / 1000
    afr = float(np.clip(air_raw / ng, *LIM["afr"]))
    air = float(np.clip(afr * ng, *LIM["sec_air"]))
    afr = air / ng
    bb_raw = inp["bb_kwh_last_hour"] + BOOST["ki"] * (BOOST["mb3_target"] - inp["mb3_now"]) / BOOST["mb3_gain_degC_per_kwh_h"]
    bb = float(np.clip(bb_raw, *LIM["bb_kwh_h"]))
    flags = []
    if ng_raw > LIM["ng_scm"][1] and bb < inp["bb_kwh_last_hour"]:
        bb = float(inp["bb_kwh_last_hour"]); flags.append("NG at its historical maximum: barrier boost not reduced this hour")
    for key, lim_key, label in [("ncv", "ncv", "NCV"), ("draw_t", "draw_t", "draw"), ("optical_prev24h", "opt_temp", "optical crown"),
                                ("mb3_now", "mb3_temp", "MB3")]:
        lo, hi = LIM[lim_key]
        if not lo <= inp[key] <= hi:
            flags.append(f"input {label} = {inp[key]:.1f} is outside its historical range {lo:.1f}-{hi:.1f}")
    if not LIM["gas_day_gcal"][0] <= target <= LIM["gas_day_gcal"][1]:
        flags.append(f"daily gas target {target:.1f} Gcal is outside its historical range (setpoints are still held inside limits)")
    rows = [
        ("NG flow setpoint", "SCM / 15 min", ng_raw, ng, "ng_scm", ng_now, 1),
        ("Secondary air", "SCM / 15 min", air_raw, air, "sec_air", air_now, 0),
        ("Air-fuel ratio", "-", air_raw / ng, afr, "afr", (air_now / ng_now) if ng_now else np.nan, 2),
        ("Barrier boost, next hour", "kWh / h", bb_raw, bb, "bb_kwh_h", inp["bb_kwh_last_hour"], 0),
        ("Barrier boost, per 15 min", "kWh / 15 min", bb_raw / 4, bb / 4, "bb_kwh_h_15", inp["bb_kwh_last_hour"] / 4, 1),
    ]
    out = []
    for name, unit, raw_v, val, lk, cur, dec in rows:
        lo, hi = (v / 4 for v in LIM["bb_kwh_h"]) if lk == "bb_kwh_h_15" else LIM[lk]
        held = not np.isclose(raw_v, val)
        out.append({"setpoint": name, "unit": unit, "current": cur, "recommended": round(val, dec), "change": val - cur if pd.notna(cur) else np.nan,
                    "historical low (P1)": round(lo, dec), "historical high (P99)": round(hi, dec),
                    "within historical range": "yes" if lo - 1e-9 <= val <= hi + 1e-9 else "NO",
                    "held at limit": "yes" if held else "no", "_dec": dec})
    table = pd.DataFrame(out)
    table.attrs["gas_target_gcal_day"] = target
    return table, flags


rec, rec_flags = recommend(inputs)
flags = data_flags + rec_flags
print(f"Recommendation for {asof:%d %b %Y %H:%M} | daily gas-heat target {rec.attrs['gas_target_gcal_day']:.2f} Gcal/day | model {MODEL_VERSION}")
rec.drop(columns="_dec").round(2)

Recommendation for 30 Sep 2026 23:45 | daily gas-heat target 79.54 Gcal/day | model 2026-09-29_202610060711


,setpoint,unit,current,recommended,change,historical low (P1),historical high (P99),within historical range,held at limit
0,NG flow setpoint,SCM / 15 min,87.24,96.10,8.86,73.90,96.30,yes,no
1,Secondary air,SCM / 15 min,1019.52,1056.00,36.57,921.00,1104.00,yes,yes
2,Air-fuel ratio,-,11.69,10.99,-0.70,10.99,13.44,yes,yes
3,"Barrier boost, next hour",kWh / h,402.20,399.00,-3.28,114.00,610.00,yes,no
4,"Barrier boost, per 15 min",kWh / 15 min,100.55,99.70,-0.82,28.60,152.40,yes,no


## 5. Historical-limit check
Every recommended setpoint must lie inside the furnace's own P1–P99 history. A value the model would have placed outside is held at the limit and marked. Inputs outside their history are reported as warnings.

In [7]:
all_within = bool((rec["within historical range"] == "yes").all())
any_held = bool((rec["held at limit"] == "yes").any())
print("All recommended setpoints are within the historical range." if all_within else "WARNING: a setpoint is outside the historical range.")
print("Setpoints held at a historical limit:", ", ".join(rec.loc[rec["held at limit"] == "yes", "setpoint"]) or "none")
print("Warnings:", ("\n - " + "\n - ".join(flags)) if flags else "none")

All recommended setpoints are within the historical range.
Setpoints held at a historical limit: Secondary air, Air-fuel ratio
Warnings: 
 - boost data is old: last complete hour ended 30 Sep 00:00
 - optical data is old: last valid value at 28 Sep 13:45


## 6. The e-mail
Outlook-safe HTML with inline styles only: the setpoints the operators should set, the current values, the change, and the historical range. It is previewed below; nothing is sent by this cell.

In [8]:
NAVY, GREY, LINE_CLR, AMBER = "#0b2545", "#5b6b7c", "#dde5ec", "#b45309"
SEPARATOR = ";"           # what the Logic App expects between addresses


def _num(v, dec) -> str:
    return "&ndash;" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:,.{dec}f}"


def _change_cell(move, dec) -> str:
    if move is None or (isinstance(move, float) and np.isnan(move)):
        return f'<span style="color:{GREY};">&ndash;</span>'
    if abs(move) < 0.5 * 10 ** (-dec):
        return f'<span style="color:{GREY};">no change</span>'
    arrow = "&#9650;" if move > 0 else "&#9660;"
    return f'<span style="color:{NAVY};font-weight:600;">{arrow} {abs(move):,.{dec}f}</span>'


def _table(headers, rows, aligns, widths, max_width="760px") -> str:
    head = "".join(
        f'<th style="background:{NAVY};color:#ffffff;font-size:11px;font-weight:600;letter-spacing:.02em;line-height:1.35;'
        f'text-transform:uppercase;padding:8px 6px;text-align:center;border:1px solid {NAVY};width:{widths[i]};">{h}</th>'
        for i, h in enumerate(headers))
    body = ""
    for r, row in enumerate(rows):
        tint = "#ffffff" if r % 2 == 0 else "#f4f7fa"
        body += "<tr>" + "".join(
            f'<td style="background:{tint};color:#16202a;font-size:12px;padding:6px 5px;text-align:{aligns[i]};'
            f'white-space:nowrap;border:1px solid {LINE_CLR};">{c}</td>' for i, c in enumerate(row)) + "</tr>"
    return ('<table cellpadding="0" cellspacing="0" border="0" '
            f'style="border-collapse:collapse;table-layout:fixed;width:100%;max-width:{max_width};font-family:Segoe UI,Arial,sans-serif;">'
            f"<thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>")


def recommendation_message(rec: pd.DataFrame, inputs: dict, flags: list, asof: pd.Timestamp, generated_at: dt.datetime) -> dict:
    subject = f"{MAIL_SUBJECT_PREFIX} for {PLANT_CAPACITY} melter - {asof:%d %b %Y %H:%M}"
    rows = []
    for r in rec.to_dict("records"):
        d = r["_dec"]
        held = f" <span style='color:{AMBER};font-size:11px;'>(at limit)</span>" if r["held at limit"] == "yes" else ""
        rows.append([f"<b>{r['setpoint']}</b><br><span style='color:{GREY};font-size:11px;'>{r['unit']}</span>",
                     _num(r["current"], d), f'<b style="color:{NAVY};font-size:13px;">{_num(r["recommended"], d)}</b>{held}',
                     _change_cell(r["change"], d), f"{_num(r['historical low (P1)'], d)} &ndash; {_num(r['historical high (P99)'], d)}"])
    setpoints = _table(["Setpoint", "Current", "Recommended", "Change", "Historical range"], rows,
                       ["left", "center", "center", "center", "center"], ["30%", "15%", "20%", "15%", "20%"])
    used = _table(["NCV (kcal/SCM)", "Draw (t/day)", "MB3 (&deg;C)", "Barrier boost last h (kWh)", "Melter boost last h (kWh)"],
                  [[_num(inputs["ncv"], 0), _num(inputs["draw_t"], 1), _num(inputs["mb3_now"], 1),
                    _num(inputs["bb_kwh_last_hour"], 0), _num(inputs["mb_kwh_last_hour"], 0)]],
                  ["center"] * 5, ["20%"] * 5)
    within = (rec["within historical range"] == "yes").all()
    limit_line = ("These recommended setpoints are within the limits of the historical range."
                  if within and not (rec["held at limit"] == "yes").any()
                  else "These recommended setpoints are within the limits of the historical range; values marked "
                       "<i>(at limit)</i> were held at the historical limit.")
    warn = ""
    if flags:
        warn = (f'<p style="margin:14px 0 4px;color:{AMBER};font-weight:600;">Please check before applying:</p><ul style="margin:0;color:{AMBER};">'
                + "".join(f"<li>{f}</li>" for f in flags) + "</ul>")
    body = f"""<div style="font-family:Segoe UI,Arial,sans-serif;font-size:14px;color:#16202a;line-height:1.55;">
<p style="margin:0 0 4px;">SFC model recommendation set points for the <b>{PLANT_CAPACITY} melter</b>
&ndash; <b>{asof:%d %b %Y, %H:%M}</b></p>
<p style="margin:0 0 12px;color:{GREY};font-size:12.5px;">NG flow, secondary air and air-fuel ratio: set in the minute 15&ndash;20 window of the
next reversal cycle (one NG value for both ports). Barrier boost: for the next hour.</p>
{setpoints}
<p style="margin:16px 0 6px;font-size:12.5px;color:{GREY};">Plant values used</p>
{used}
<p style="margin-top:18px;">{limit_line}</p>
{warn}
<p style="margin-top:18px;">Regards,<br><b>DnA Team</b></p>
<p style="font-size:11.5px;color:{GREY};border-top:1px solid {LINE_CLR};padding-top:10px;max-width:760px;">
Recommendation generated {generated_at:%d %b %Y, %H:%M} &middot; model version {MODEL_VERSION} (trained until {model["trained_until"]}).
</p>
</div>"""
    return {"subject": subject, "body": body}


def send_mail(email_to, subject: str, body: str, email_cc=None) -> int:
    """Post one message on the Logic App's payload keys (EmailTo, Subject, Message_line1, EmailCC)."""
    if not email_to:
        raise ValueError("At least one 'To' address is required.")
    data = {"EmailTo": SEPARATOR.join(email_to), "Subject": subject, "Message_line1": body}
    if email_cc:
        data["EmailCC"] = SEPARATOR.join(email_cc)
    resp = requests.post(MAIL_URL, json=data, timeout=MAIL_TIMEOUT)
    if resp.status_code >= 400:
        raise RuntimeError(f"the mail service refused the message (HTTP {resp.status_code}): {resp.text[:300]}")
    return resp.status_code

### Preview
What the recipients would see, rendered in the notebook. Nothing is sent by this cell.

In [9]:
generated_at = dt.datetime.now()
message = recommendation_message(rec, inputs, flags, asof, generated_at)
print("To:", ", ".join(TO_EMAILS), "| Cc:", ", ".join(CC_EMAILS) or "-")
print("Subject:", message["subject"])
displayHTML(message["body"])

To: balaji.kesavan@cantier.com | Cc: rudhuvarshan.r@cantier.com
Subject: SFC Recommendation for 60 TPD melter - 30 Sep 2026 23:45


Setpoint,Current,Recommended,Change,Historical range
NG flow setpointSCM / 15 min,87.2,96.1,▲ 8.9,73.9 – 96.3
Secondary airSCM / 15 min,"1,020","1,056 (at limit)",▲ 37,"921 – 1,104"
Air-fuel ratio-,11.69,10.99 (at limit),▼ 0.70,10.99 – 13.44
"Barrier boost, next hourkWh / h",402,399,▼ 3,114 – 610
"Barrier boost, per 15 minkWh / 15 min",100.5,99.7,▼ 0.8,28.6 – 152.4
NCV (kcal/SCM),Draw (t/day),MB3 (°C),Barrier boost last h (kWh),Melter boost last h (kWh)
"8,621",61.1,"1,320.9",402,307


### Send
Set `SEND_MAIL = True` in the settings cell to let the message through. Every issued recommendation is also appended to `output/recommendation_log.csv`, so a setpoint on the floor can always be traced back to its model version and inputs (and compared with what was actually set).

In [10]:
status = None
if SEND_MAIL:
    status = send_mail(TO_EMAILS, message["subject"], message["body"], CC_EMAILS)
    print(f"sent -> {', '.join(TO_EMAILS)} [{status}]")
else:
    print(f"not sent (SEND_MAIL is False): {message['subject']}")

log = {"generated_at": generated_at.isoformat(timespec="seconds"), "recommendation_for": str(asof), "model_version": MODEL_VERSION,
       **{f"in_{k}": (str(v) if k == "date" else v) for k, v in inputs.items()},
       **{f"rec_{r.setpoint.lower().replace(' ', '_').replace(',', '')}_{r.unit.replace(' ', '').replace('/', 'per')}": r.recommended for r in rec.itertuples()},
       "gas_target_gcal_day": rec.attrs["gas_target_gcal_day"], "all_within_historical_range": all_within,
       "flags": " | ".join(flags), "mail_sent": bool(SEND_MAIL), "mail_status": status, "to": SEPARATOR.join(TO_EMAILS)}
log_path = OUTPUT_DIR / "recommendation_log.csv"
pd.DataFrame([log]).to_csv(log_path, mode="a", header=not log_path.exists(), index=False)
# In Databricks, the same row can go to a Delta table instead:
# spark.createDataFrame(pd.DataFrame([log])).write.mode("append").saveAsTable("prod_pgp_mfg.cantier.sfc_recommendation_log")
print("logged ->", log_path)

not sent (SEND_MAIL is False): SFC Recommendation for 60 TPD melter - 30 Sep 2026 23:45
logged -> output/recommendation_log.csv
